## 神经网络模型预测全国人均衣着消费支出

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os

# 设置随机种子以确保结果可重现
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seed(42)

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 数据准备
data = {
    'year': [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023],
    'value': [849.48, 1004.20, 1130.72, 1131.24, 1133.99, 1194.36, 1231.23, 1264.94, 1331.27, 1379.06,1249.69,1419.54,1344.53,1469.26]
}

df = pd.DataFrame(data)

# 添加趋势特征
def add_trend_features(years, values):
    x = np.array(years).reshape(-1, 1)
    y = np.array(values)
    
    from sklearn.linear_model import LinearRegression
    trend_model = LinearRegression()
    trend_model.fit(x, y)
    trend_component = trend_model.predict(x)
    
    # 计算增长率特征
    growth_rates = []
    for i in range(1, len(values)):
        growth_rate = (values[i] - values[i-1]) / values[i-1]
        growth_rates.append(growth_rate)
    # 第一个位置补0
    growth_rates = [0] + growth_rates
    
    return trend_component, np.array(growth_rates)

# 准备增强特征
years = df['year'].values
values = df['value'].values
trend_component, growth_rates = add_trend_features(years, values)

# 创建多维特征
features = np.column_stack([
    values.reshape(-1, 1),  # 原始值
    trend_component.reshape(-1, 1),  # 趋势成分
    growth_rates.reshape(-1, 1),  # 增长率
    np.arange(len(values)).reshape(-1, 1)  # 时间序号
])

# 数据标准化
scalers = {}
scaled_features = np.zeros_like(features)
for i in range(features.shape[1]):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_features[:, i] = scaler.fit_transform(features[:, i].reshape(-1, 1)).flatten()
    scalers[i] = scaler

def create_enhanced_dataset(data, time_step=3):
    X, y = [], []
    for i in range(len(data)-time_step):
        X.append(data[i:(i+time_step)])  # 多特征输入
        y.append(data[i + time_step, 0])  # 只预测原始值
    return np.array(X), np.array(y)

# 设置时间步长
time_step = 3
X, y = create_enhanced_dataset(scaled_features, time_step)

# 分割数据
train_size = int(len(X) * 0.7)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"训练集形状: {X_train.shape}, 测试集形状: {X_test.shape}")

# LSTM模型 
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(time_step, X.shape[2]), 
         kernel_initializer=tf.keras.initializers.glorot_uniform(seed=42)),
    Dropout(0.3, seed=42),
    LSTM(64, return_sequences=True, 
         kernel_initializer=tf.keras.initializers.glorot_uniform(seed=43)),
    Dropout(0.3, seed=43),
    LSTM(32, kernel_initializer=tf.keras.initializers.glorot_uniform(seed=44)),
    Dropout(0.2, seed=44),
    Dense(16, activation='relu', kernel_initializer=tf.keras.initializers.glorot_uniform(seed=45)),
    Dense(1, kernel_initializer=tf.keras.initializers.glorot_uniform(seed=46))
])


optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mean_squared_error')

print("=== LSTM神经网络模型结构 ===")
model.summary()

# 训练模型 
history = model.fit(X_train, y_train, 
                   epochs=200, 
                   batch_size=2, 
                   validation_data=(X_test, y_test),
                   verbose=1,
                   shuffle=False)  # 重要：关闭shuffle


def forecast_future_with_trend(model, last_sequence, steps, scalers, original_data):

    predictions = []
    current_sequence = last_sequence.copy()
    

    years = np.arange(len(original_data))
    values = original_data
    trend_coef = np.polyfit(years, values, 1)[0] 
    
    for step in range(steps):
        # 预测下一个值 
        next_pred_scaled = model.predict(current_sequence.reshape(1, time_step, -1), 
                                       verbose=0, batch_size=1)
        
      
        next_pred_original = scalers[0].inverse_transform(next_pred_scaled)[0, 0]
        
        trend_adjustment = trend_coef * 0.6  
        adjusted_pred = next_pred_original + trend_adjustment
        
        # 更新序列用于下一次预测
        new_features = []
        
    
        new_features.append(adjusted_pred)
        
  
        last_trend = scalers[1].inverse_transform(current_sequence[-1:, 1].reshape(-1, 1))[0, 0]
        new_trend = last_trend + trend_coef
        new_features.append(new_trend)
        
   
        last_growth = scalers[2].inverse_transform(current_sequence[-1:, 2].reshape(-1, 1))[0, 0]
        new_growth = max(last_growth, 0.01)  # 保持最小正增长
        new_features.append(new_growth)
        

        last_time = scalers[3].inverse_transform(current_sequence[-1:, 3].reshape(-1, 1))[0, 0]
        new_time = last_time + 1
        new_features.append(new_time)
        

        new_features_scaled = []
        for i, feature in enumerate(new_features):
            scaled_feature = scalers[i].transform(np.array([[feature]]))[0, 0]
            new_features_scaled.append(scaled_feature)
        

        current_sequence = np.roll(current_sequence, -1, axis=0)
        current_sequence[-1] = new_features_scaled
        
        predictions.append(adjusted_pred)
    
    return np.array(predictions)

# 进行预测
last_sequence = scaled_features[-time_step:]
future_steps = 2100 - 2023
future_predictions = forecast_future_with_trend(model, last_sequence, future_steps, scalers, values)

# 创建完整结果
historical_years = df['year'].values
future_years = np.arange(2024, 2101)
all_years = np.concatenate([historical_years, future_years])

# 获取历史拟合值 
train_predict = model.predict(X_train, verbose=0, batch_size=1)
train_predict_original = scalers[0].inverse_transform(train_predict.reshape(-1, 1)).flatten()

test_predict = model.predict(X_test, verbose=0, batch_size=1)
test_predict_original = scalers[0].inverse_transform(test_predict.reshape(-1, 1)).flatten()

# 组合所有预测值
all_predictions = np.concatenate([
    values[:time_step],  # 前time_step个实际值
    train_predict_original,
    test_predict_original,
    future_predictions
])

results_nn_enhanced = pd.DataFrame({
    'year': all_years,
    'predicted_value': all_predictions[:len(all_years)]
})


# 计算评估指标
def normalize_data_for_evaluation(actual_values, predicted_values, original_values):
  
    min_val = np.min(original_values)
    max_val = np.max(original_values)
    
    actual_normalized = (actual_values - min_val) / (max_val - min_val)
    predicted_normalized = (predicted_values - min_val) / (max_val - min_val)
    
    return actual_normalized, predicted_normalized


y_test_actual = values[train_size+time_step:]
y_test_pred = test_predict_original

y_test_actual_norm, y_test_pred_norm = normalize_data_for_evaluation(
    y_test_actual, y_test_pred, values
)


mse_norm = mean_squared_error(y_test_actual_norm, y_test_pred_norm)
rmse_norm = np.sqrt(mse_norm)
mae_norm = mean_absolute_error(y_test_actual_norm, y_test_pred_norm)
r2_test_norm = r2_score(y_test_actual_norm, y_test_pred_norm)

print("\n=== LSTM神经网络模型评估指标 ===")
print(f"测试集均方误差 (MSE): {mse_norm:.6f}")
print(f"测试集根均方误差 (RMSE): {rmse_norm:.6f}")
print(f"测试集平均绝对误差 (MAE): {mae_norm:.6f}")
print(f"测试集决定系数 (R²): {r2_test_norm:.6f}")

# 计算整体R²（所有历史拟合值）
historical_actual = values[time_step:] 
historical_fitted = np.concatenate([train_predict_original, test_predict_original])


min_len = min(len(historical_actual), len(historical_fitted))
historical_actual = historical_actual[:min_len]
historical_fitted = historical_fitted[:min_len]


historical_actual_norm, historical_fitted_norm = normalize_data_for_evaluation(
    historical_actual, historical_fitted, values
)


overall_r2_norm = r2_score(historical_actual_norm, historical_fitted_norm)
overall_mse_norm = mean_squared_error(historical_actual_norm, historical_fitted_norm)
overall_rmse_norm = np.sqrt(overall_mse_norm)
overall_mae_norm = mean_absolute_error(historical_actual_norm, historical_fitted_norm)

print(f"\n=== 整体拟合效果评估===")
print(f"历史实际值数量: {len(historical_actual)}")
print(f"历史拟合值数量: {len(historical_fitted)}")
print(f"整体拟合决定系数 (R²): {overall_r2_norm:.6f}")
print(f"整体均方误差 (MSE): {overall_mse_norm:.6f}")
print(f"整体根均方误差 (RMSE): {overall_rmse_norm:.6f}")
print(f"整体平均绝对误差 (MAE): {overall_mae_norm:.6f}")

# ============= 原始终点预测值保持不变 =============
print(f"\n=== 原始预测结果 ===")
print(f"最终预测2100年值: {future_predictions[-1]:.2f}")

# 验证结果一致性
print(f"\n=== 结果一致性验证 ===")
print(f"模型预测2100年值: {future_predictions[-1]:.2f}")
print(f"训练集第一个预测值: {train_predict_original[0]:.2f}")
print(f"测试集第一个预测值: {test_predict_original[0]:.2f}")

# 可视化
plt.figure(figsize=(15, 12))

# 损失曲线
plt.subplot(3, 1, 1)
plt.plot(history.history['loss'], label='训练损失')
plt.plot(history.history['val_loss'], label='验证损失')
plt.title('增强LSTM模型损失曲线 (确定性训练)')
plt.ylabel('损失')
plt.xlabel('轮次')
plt.legend()
plt.grid(True, alpha=0.3)

# 预测结果 - 使用原始预测值
plt.subplot(3, 1, 2)
plt.plot(historical_years, values, 'bo-', label='实际数据', linewidth=2, markersize=6)
plt.plot(results_nn_enhanced['year'], results_nn_enhanced['predicted_value'], 'r-', 
         label='增强LSTM预测', linewidth=2)
plt.axvline(x=2023, color='gray', linestyle='--', alpha=0.7, label='预测起点')
plt.xlabel('年份')
plt.ylabel('数值')
plt.title(f'增强LSTM神经网络预测\n归一化整体R²={overall_r2_norm:.3f}, 归一化测试集R²={r2_test_norm:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)

# 拟合效果散点图 - 使用原始数据
plt.subplot(3, 1, 3)
plt.scatter(historical_actual, historical_fitted, alpha=0.6, color='blue')
plt.plot([historical_actual.min(), historical_actual.max()], 
         [historical_actual.min(), historical_fitted.max()], 'r--', linewidth=2)
plt.xlabel('实际值')
plt.ylabel('拟合值')
plt.title(f'历史数据拟合效果 (归一化R²={overall_r2_norm:.3f})')
plt.grid(True, alpha=0.3)

# 添加拟合线方程
z = np.polyfit(historical_actual, historical_fitted, 1)
p = np.poly1d(z)
plt.plot(historical_actual, p(historical_actual), "g--", alpha=0.8, 
         label=f'拟合线: y = {z[0]:.2f}x + {z[1]:.2f}')
plt.legend()

plt.tight_layout()
plt.show()

print("\n增强LSTM模型部分预测结果:")
print(results_nn_enhanced[results_nn_enhanced['year'] % 10 == 0].tail(10))

# 保存预测结果到CSV文件
results_nn_enhanced.to_csv('lstm_predictions_deterministic.csv', index=False, encoding='utf-8-sig')
print("\n预测结果已保存到: lstm_predictions_deterministic.csv")

# 保存每10年的数据
decadal_data = results_nn_enhanced[results_nn_enhanced['year'] % 10 == 0]
decadal_data.to_csv('lstm_predictions_decadal_deterministic.csv', index=False, encoding='utf-8-sig')
print("十年间隔数据已保存到: lstm_predictions_decadal_deterministic.csv")

print(f"\n预测数据形状: {results_nn_enhanced.shape}")
print(f"数据时间范围: {results_nn_enhanced['year'].min()}年 - {results_nn_enhanced['year'].max()}年")

# 验证可重现性
print(f"\n=== 可重现性验证 ===")
print("重新运行此代码将得到完全相同的预测结果")
print(f"验证值 - 2100年预测: {future_predictions[-1]:.2f}")
print(f"验证值 - 归一化整体R²: {overall_r2_norm:.6f}")
